# 7.1 Creating a simple autoencoder in Keras

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Load the MNIST dataset and normalize it
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = x_train.reshape((len(x_train), 28, 28, 1))
x_test = x_test.reshape((len(x_test), 28, 28, 1))

# Encoder
input_img = layers.Input(shape=(28, 28, 1))
x = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(input_img)
x = layers.MaxPooling2D((2, 2), padding="same")(x)
x = layers.Conv2D(8, (3, 3), activation="relu", padding="same")(x)
x = layers.MaxPooling2D((2, 2), padding="same")(x)
encoded = layers.Conv2D(8, (3, 3), activation="relu", padding="same")(x)

# Decoder
x = layers.Conv2D(8, (3, 3), activation="relu", padding="same")(encoded)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(x)
x = layers.UpSampling2D((2, 2))(x)
decoded = layers.Conv2D(1, (3, 3), activation="sigmoid", padding="same")(x)

# Autoencoder model
autoencoder = models.Model(input_img, decoded)
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")

# Train the autoencoder
autoencoder.fit(
    x_train, x_train, epochs=50, batch_size=256, validation_data=(x_test, x_test)
)

Epoch 1/50
  9/235 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 0.6609 

E0000 00:00:1791983909.965307   50381 util.cc:131] oneDNN supports DT_INT32 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - loss: 0.2284 - val_loss: 0.1112
Epoch 2/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0999 - val_loss: 0.0916
Epoch 3/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0893 - val_loss: 0.0856
Epoch 4/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0849 - val_loss: 0.0827
Epoch 5/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 0.0823 - val_loss: 0.0804
Epoch 6/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0805 - val_loss: 0.0790
Epoch 7/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0794 - val_loss: 0.0782
Epoch 8/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0784 - val_loss: 0.0772
Epoch 9/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0777 - val_loss: 0.0767
Epoch 10/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0771 - val_loss: 0.0762
Epoch 11/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0766 - val_loss: 0.0759
Epoch 12/50
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/ste

# 7.2 Implementation of a variational autoencoder in Keras

In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf
import numpy as np

# Sampling function for the latent space
def sampling(args):
    z_mean, z_log_var = args
    batch = tf.shape(z_mean)[0]
    dim = tf.shape(z_mean)[1]
    epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

class VAELossLayer(layers.Layer):
    def call(self, inputs):
        original, reconstruction, z_mean, z_log_var = inputs
        reconstruction_loss = tf.keras.ops.binary_crossentropy(original, reconstruction)
        reconstruction_loss = tf.keras.ops.sum(reconstruction_loss, axis=(1, 2))
        kl_loss = -0.5 * tf.keras.ops.sum(
            1 + z_log_var - tf.keras.ops.square(z_mean) - tf.keras.ops.exp(z_log_var),
            axis=-1,
        )
        self.add_loss(tf.keras.ops.mean(reconstruction_loss + kl_loss))
        return reconstruction

# Encoder
latent_dim = 2
inputs = layers.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(inputs)
x = layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Flatten()(x)
x = layers.Dense(16, activation="relu")(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)

# Latent space sampling
z = layers.Lambda(sampling, output_shape=(latent_dim,), name="z")([z_mean, z_log_var])

# Decoder
decoder_input = layers.Input(shape=(latent_dim,))
x = layers.Dense(7 * 7 * 64, activation="relu")(decoder_input)
x = layers.Reshape((7, 7, 64))(x)
x = layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(x)
decoder_output = layers.Conv2DTranspose(1, 3, activation="sigmoid", padding="same")(x)

# VAE model
encoder = models.Model(inputs, [z_mean, z_log_var, z], name="encoder")
decoder = models.Model(decoder_input, decoder_output, name="decoder")
vae_output = decoder(encoder(inputs)[2])
vae_output = VAELossLayer(name="vae_loss")([inputs, vae_output, z_mean, z_log_var])
vae = models.Model(inputs, vae_output, name="vae")

vae.compile(optimizer="adam")

# Train the VAE
vae.fit(x_train, x_train, epochs=50, batch_size=128, validation_data=(x_test, x_test))

Epoch 1/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - loss: 217.0349 - val_loss: 191.9529
Epoch 2/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - loss: 181.4760 - val_loss: 172.6952
Epoch 3/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - loss: 169.8999 - val_loss: 168.0846
Epoch 4/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - loss: 166.3252 - val_loss: 165.1489
Epoch 5/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - loss: 163.8620 - val_loss: 163.0415
Epoch 6/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - loss: 162.1456 - val_loss: 161.6872
Epoch 7/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - loss: 160.9232 - val_loss: 160.7468
Epoch 8/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - loss: 159.8810 - val_loss: 159.6772
Epoch 9/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - loss: 158.9576 - val_loss: 158.7903
Epoch 10/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - loss: 158.3193 - val_loss: 158.1604
Epoch 11/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - loss: 157.7094 - val_loss: 157.